# MT v2: Improved Seq2Seq + Beam Search + Prob Ensemble

In [ ]:

# pip install torch pandas sentencepiece sacrebleu tqdm


In [ ]:

import torch, random, os, zipfile
import torch.nn as nn
import torch.optim as optim
from pathlib import Path
import pandas as pd
import sentencepiece as spm
from tqdm import tqdm

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

PAD, UNK, BOS, EOS = 0,1,2,3


## Config

In [ ]:

VOCAB_SIZE=8000
EMB_SIZE=256
HID_SIZE=512
MAX_LEN=100

EPOCHS=36
LR=2e-4
BATCH_SIZE=64

N_MODELS=3
SEEDS=[42,36,72]


## Beam Search

In [ ]:

def beam_search(model, src, sp_tgt, beam_size=3):
    with torch.no_grad():
        enc_out, hidden = model.encoder(src)
        beams = [([BOS], 0.0, hidden)]

        for _ in range(MAX_LEN):
            new_beams=[]
            for tokens, score, h in beams:
                if tokens[-1]==EOS:
                    new_beams.append((tokens,score,h))
                    continue

                inp=torch.tensor([[tokens[-1]]],device=DEVICE)
                logits,h2=model.decoder(inp,h,enc_out)
                probs=torch.log_softmax(logits,dim=1)

                topk=torch.topk(probs,beam_size,dim=1)
                for i in range(beam_size):
                    tok=topk.indices[0,i].item()
                    sc=topk.values[0,i].item()
                    new_beams.append((tokens+[tok], score+sc, h2))

            beams=sorted(new_beams,key=lambda x:x[1]/(len(x[0])**0.7),reverse=True)[:beam_size]

        best=beams[0][0]
        if EOS in best:
            best=best[1:best.index(EOS)]
        else:
            best=best[1:]
        return sp_tgt.DecodeIds(best)


## Probability Ensemble

In [ ]:

def ensemble_decode(models, src, sp_src, sp_tgt):
    with torch.no_grad():
        src_ids=[BOS]+sp_src.EncodeAsIds(src)[:MAX_LEN-2]+[EOS]
        src_tensor=torch.tensor(src_ids,device=DEVICE).unsqueeze(0)

        encs=[]
        hiddens=[]
        for m in models:
            eo,h=m.encoder(src_tensor)
            encs.append(eo)
            hiddens.append(h)

        tokens=[BOS]

        for _ in range(MAX_LEN):
            probs_sum=None

            for i,m in enumerate(models):
                inp=torch.tensor([[tokens[-1]]],device=DEVICE)
                logits,h=m.decoder(inp,hiddens[i],encs[i])
                p=torch.softmax(logits,dim=1)

                probs_sum = p if probs_sum is None else probs_sum+p
                hiddens[i]=h

            probs_avg=probs_sum/len(models)
            next_tok=probs_avg.argmax(dim=1).item()

            if next_tok==EOS:
                break
            tokens.append(next_tok)

        return sp_tgt.DecodeIds(tokens[1:])
